In [75]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import re
import io

display(HTML("""
<style>
.jp-OutputArea-output, .widget-label, .widget-html-content {
    font-size: 14px;
}
</style>
"""))

# =========================
# GLOBALS
# =========================
df_trim = None
df_max_sel_global = None
df_min_sel_global = None
mirror_shift_global = 0.0

image12_data_global = None
image13_data_global = None
image0_data_global = None
image11_raw_global = None

loaded_filename_global = None

begin_info_global = None
end_info_global = None
sampling_frequency_global = None
image0_best_percentage_global = None
image0_best_std_global = None

MARKER_SIZE = 3
SMOOTHING_IMAGE12 = 5.0

extrema_range_percent_global = 0.01

DATA_SELECTION_PERCENT_DEFAULT = 80.0
data_selection_percent_global = DATA_SELECTION_PERCENT_DEFAULT

image12_xmax_global = None
image12_ymax_global = None

image13_xmax_global = None
image13_ymax_global = None

image13_avg_start_global = None
image13_avg_end_global = None
image13_avg_value_global = None
image13_std_value_global = None

STD_FRACTION_GLOBAL = 0.60

ALLOWED_FREQUENCY_CENTERS_GLOBAL = [0.5, 1.0, 2.0]
FREQ_LOCAL_HALF_WIDTH_GLOBAL = 0.10
FREQ_FINE_SCAN_HALF_WIDTH_GLOBAL = 0.00005


# =========================
# HELPERS
# =========================
def get_auto_threshold_from_sampling_frequency(sampling_frequency_hz):
    return 0.01 if sampling_frequency_hz >= 20.0 else 0.05


def find_header_row_from_text(text, key_text="Elapsed Time"):
    for i, line in enumerate(text.splitlines()):
        if key_text in line:
            return i
    return None


def find_sampling_frequency_in_text(text, key_text="Elapsed Time"):
    header_row = find_header_row_from_text(text, key_text=key_text)
    if header_row is None:
        raise ValueError("Header row not found, so sampling frequency cannot be detected.")

    header_lines = text.splitlines()[:header_row + 1]
    header_text = "\n".join(header_lines)

    hz_matches = re.findall(r"(\d+(?:\.\d+)?)\s*Hz\b", header_text, flags=re.IGNORECASE)
    if hz_matches:
        return float(hz_matches[-1])

    freq_matches = re.findall(
        r"(?:sampling\s*frequency|logging\s*frequency|sample\s*rate|sampling\s*rate)\s*[:=]?\s*(\d+(?:\.\d+)?)",
        header_text,
        flags=re.IGNORECASE
    )
    if freq_matches:
        return float(freq_matches[-1])

    raise ValueError("Sampling frequency not found in header.")


def load_sensor_file_from_bytes(file_bytes):
    if isinstance(file_bytes, memoryview):
        file_bytes = file_bytes.tobytes()

    text = file_bytes.decode("utf-8", errors="ignore")

    header_row = find_header_row_from_text(text)
    if header_row is None:
        raise ValueError("Header not found.")

    df = pd.read_csv(
        io.StringIO(text),
        skiprows=header_row + 1,
        names=["Time", "Elapsed Time (s)", "Friction Force (N)"],
        sep=None,
        engine="python",
        on_bad_lines="skip"
    )

    df = df.dropna(how="all").copy()
    df["Elapsed Time (s)"] = pd.to_numeric(df["Elapsed Time (s)"], errors="coerce")
    df["Friction Force (N)"] = pd.to_numeric(df["Friction Force (N)"], errors="coerce")
    df = df.dropna(subset=["Elapsed Time (s)", "Friction Force (N)"]).reset_index(drop=True)

    return df, text


def get_sign_groups(y):
    y = np.asarray(y, dtype=float)
    signs = np.where(y >= 0, 1, -1)

    groups = []
    i = 0
    n = len(signs)

    while i < n:
        start = i
        sign = signs[i]

        while i < n and signs[i] == sign:
            i += 1

        end = i - 1
        groups.append({
            "start": start,
            "end": end,
            "sign": sign,
            "length": end - start + 1
        })

    return groups


def trim_beginning_by_first_20_groups(df, sampling_frequency_hz, max_groups=20):
    y = df["Friction Force (N)"].to_numpy(dtype=float)
    groups = get_sign_groups(y)

    if len(groups) < 1:
        raise ValueError("No sign groups found.")

    groups_20 = groups[:min(max_groups, len(groups))]
    lengths = np.array([g["length"] for g in groups_20], dtype=float)

    target = float(np.median(lengths))
    tolerance = max(1.0, 0.20 * target)

    lower = target - tolerance
    upper = target + tolerance

    selected_group_index = None
    selected_group = None

    for i in range(len(groups_20) - 1, -1, -1):
        g = groups_20[i]
        if g["length"] < lower or g["length"] > upper:
            selected_group_index = i
            selected_group = g
            break

    if selected_group is None:
        selected_group_index = 0
        selected_group = groups_20[0]

    keep_start_idx = max(selected_group["start"], selected_group["end"] - 2)
    df_out = df.iloc[keep_start_idx:].copy().reset_index(drop=True)

    info = {
        "sampling_frequency_hz": sampling_frequency_hz,
        "target_group_length": target,
        "allowed_min": lower,
        "allowed_max": upper,
        "groups_checked": groups_20,
        "selected_group_1based": selected_group_index + 1,
        "selected_group": selected_group,
        "keep_start_idx_original": keep_start_idx
    }
    return df_out, info


def trim_end_by_last_20_groups(df, sampling_frequency_hz, max_groups=20):
    y = df["Friction Force (N)"].to_numpy(dtype=float)
    groups = get_sign_groups(y)

    if len(groups) < 1:
        raise ValueError("No sign groups found for last trim.")

    groups_last20 = groups[-min(max_groups, len(groups)):]
    lengths = np.array([g["length"] for g in groups_last20], dtype=float)

    target = float(np.median(lengths))
    tolerance = max(1.0, 0.20 * target)

    lower = target - tolerance
    upper = target + tolerance

    selected_group_index = None
    selected_group = None

    for i, g in enumerate(groups_last20):
        if g["length"] < lower or g["length"] > upper:
            selected_group_index = i
            selected_group = g
            break

    if selected_group is None:
        df_out = df.copy().reset_index(drop=True)
        info = {
            "sampling_frequency_hz": sampling_frequency_hz,
            "target_group_length": target,
            "allowed_min": lower,
            "allowed_max": upper,
            "groups_checked": groups_last20,
            "selected_group_1based": None,
            "selected_group": None,
            "remove_start_idx_original": None
        }
        return df_out, info

    end_keep_idx = selected_group["start"] - 1
    if end_keep_idx < 0:
        raise ValueError("Last trim would remove all data.")

    df_out = df.iloc[:end_keep_idx + 1].copy().reset_index(drop=True)

    info = {
        "sampling_frequency_hz": sampling_frequency_hz,
        "target_group_length": target,
        "allowed_min": lower,
        "allowed_max": upper,
        "groups_checked": groups_last20,
        "selected_group_1based": selected_group_index + 1,
        "selected_group": selected_group,
        "remove_start_idx_original": selected_group["start"]
    }
    return df_out, info


def trim_by_group_rule(df, sampling_frequency_hz):
    df_after_begin, begin_info = trim_beginning_by_first_20_groups(df, sampling_frequency_hz, max_groups=20)
    df_out, end_info = trim_end_by_last_20_groups(df_after_begin, sampling_frequency_hz, max_groups=20)
    return df_out, begin_info, end_info


def normalize_signal(y):
    y0 = y - np.mean(y)
    m = np.max(np.abs(y0))
    if m == 0:
        return y0
    return y0 / m


def best_phase_for_frequency(t, y_norm, freq):
    weights = np.ones_like(t)
    weights[t <= 2.0] = 3.0

    def phase_error(phase):
        ref = np.cos(2 * np.pi * freq * t - phase)
        return np.mean(weights * (y_norm - ref) ** 2)

    res = minimize_scalar(
        phase_error,
        bounds=(-np.pi, np.pi),
        method="bounded",
        options={"xatol": 1e-12}
    )
    return res.x, res.fun


def detect_frequency_and_phase(df, data_selection_fraction=0.80):
    t_all = df["Elapsed Time (s)"].to_numpy(dtype=float)
    y_all = df["Friction Force (N)"].to_numpy(dtype=float)

    if len(t_all) < 100:
        raise ValueError("Not enough data points.")

    if not (0 < data_selection_fraction <= 1):
        raise ValueError("Data Selection must be > 0 and <= 1.")

    n = len(t_all)
    i1 = int((1.0 - data_selection_fraction) * n)

    t = t_all[i1:].copy()
    y = y_all[i1:].copy()
    y_norm = normalize_signal(y)

    def freq_error(freq):
        _, err = best_phase_for_frequency(t, y_norm, freq)
        return err

    best_freq = None
    best_err = np.inf

    for center in ALLOWED_FREQUENCY_CENTERS_GLOBAL:
        lower = max(0.01, center - FREQ_LOCAL_HALF_WIDTH_GLOBAL)
        upper = center + FREQ_LOCAL_HALF_WIDTH_GLOBAL

        res_local = minimize_scalar(
            freq_error,
            bounds=(lower, upper),
            method="bounded",
            options={"xatol": 1e-12}
        )

        if res_local.fun < best_err:
            best_err = res_local.fun
            best_freq = res_local.x

    freq_best = best_freq
    y_full_norm = normalize_signal(y_all)
    phase_best, _ = best_phase_for_frequency(t_all, y_full_norm, freq_best)

    return freq_best, phase_best


def smooth_small(y, window=5):
    y = np.asarray(y, dtype=float)
    if len(y) < 3 or window <= 1:
        return y.copy()
    if window % 2 == 0:
        window += 1
    if window > len(y):
        window = len(y) if len(y) % 2 == 1 else len(y) - 1
    if window < 3:
        return y.copy()
    k = np.ones(window) / window
    ypad = np.pad(y, (window // 2, window // 2), mode="edge")
    return np.convolve(ypad, k, mode="valid")


def extract_transition_times(t, y):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(t) < 2:
        return np.array([], dtype=float)

    y_s = smooth_small(y, window=5)
    s = np.where(y_s >= 0, 1, -1)

    trans = []
    for i in range(len(s) - 1):
        if s[i] != s[i + 1]:
            y1 = y_s[i]
            y2 = y_s[i + 1]
            t1 = t[i]
            t2 = t[i + 1]

            if y2 != y1:
                tz = t1 - y1 * (t2 - t1) / (y2 - y1)
            else:
                tz = 0.5 * (t1 + t2)

            trans.append(tz)

    return np.array(trans, dtype=float)


def model_zero_crossings(freq, phase, tmin, tmax):
    k0 = int(np.floor((2 * np.pi * freq * tmin - phase - np.pi / 2) / np.pi)) - 2
    k1 = int(np.ceil((2 * np.pi * freq * tmax - phase - np.pi / 2) / np.pi)) + 2

    ts = []
    for k in range(k0, k1 + 1):
        tz = (phase + np.pi / 2 + k * np.pi) / (2 * np.pi * freq)
        if tmin <= tz <= tmax:
            ts.append(tz)

    return np.array(ts, dtype=float)


def branch_waveform_from_data(t, y):
    y0 = y - np.median(y)
    y0 = smooth_small(y0, window=5)
    return np.where(y0 >= 0, 1.0, -1.0)


def phase_transition_score(phase, t_seg, branch_seg, trans_times, freq):
    ref_branch = np.where(np.cos(2 * np.pi * freq * t_seg - phase) >= 0, 1.0, -1.0)
    mismatch = np.mean(ref_branch != branch_seg)

    model_trans = model_zero_crossings(freq, phase, t_seg.min(), t_seg.max())

    if len(trans_times) == 0 or len(model_trans) == 0:
        trans_err = 1e3
    else:
        d1 = [np.min(np.abs(model_trans - tt)) for tt in trans_times]
        d2 = [np.min(np.abs(trans_times - tt)) for tt in model_trans]
        trans_err = 0.5 * (np.mean(d1) + np.mean(d2))

    score = trans_err + 0.2 * mismatch
    return score


def detect_phase_from_transitions(df, freq, n_cycles=3):
    t = df["Elapsed Time (s)"].to_numpy(dtype=float)
    y = df["Friction Force (N)"].to_numpy(dtype=float)

    if len(t) < 5:
        return 0.0

    t_end = min(t.max(), n_cycles / freq)
    mask = t <= t_end

    t_seg = t[mask]
    y_seg = y[mask]

    if len(t_seg) < 5:
        t_seg = t
        y_seg = y

    branch_seg = branch_waveform_from_data(t_seg, y_seg)
    trans_times = extract_transition_times(t_seg, y_seg - np.median(y_seg))

    phases = np.linspace(0, 2 * np.pi, 1440, endpoint=False)
    scores = np.array([phase_transition_score(ph, t_seg, branch_seg, trans_times, freq) for ph in phases])
    best_i = int(np.argmin(scores))
    phase0 = phases[best_i]

    step = 2 * np.pi / 1440

    def wrapped_score(ph):
        ph = ph % (2 * np.pi)
        return phase_transition_score(ph, t_seg, branch_seg, trans_times, freq)

    res = minimize_scalar(
        wrapped_score,
        bounds=(phase0 - 3 * step, phase0 + 3 * step),
        method="bounded",
        options={"xatol": 1e-12}
    )

    return res.x % (2 * np.pi)


def select_friction_near_ref_extrema(df, freq, phase, extrema_range_percent=None):
    if extrema_range_percent is None:
        extrema_range_percent = extrema_range_percent_global

    t = df["Elapsed Time (s)"].to_numpy(dtype=float)
    ref = np.cos(2 * np.pi * freq * t - phase)

    max_threshold = 1.0 - extrema_range_percent
    min_threshold = -1.0 + extrema_range_percent

    max_idx = np.where(ref >= max_threshold)[0]
    min_idx = np.where(ref <= min_threshold)[0]

    if 0 in max_idx and 0 in min_idx:
        min_idx = min_idx[min_idx != 0]

    df_max = df.iloc[max_idx].copy().reset_index(drop=True) if len(max_idx) > 0 else df.iloc[[]].copy()
    df_min = df.iloc[min_idx].copy().reset_index(drop=True) if len(min_idx) > 0 else df.iloc[[]].copy()

    return df_max, df_min


def compute_mirror_shift(df_max, df_min):
    n = min(len(df_max), len(df_min))
    if n < 1:
        return 0.0

    y_max = df_max["Friction Force (N)"].to_numpy(dtype=float)[:n]
    y_min = df_min["Friction Force (N)"].to_numpy(dtype=float)[:n]
    return -0.5 * np.mean(y_max + y_min)


def build_absolute_shifted_raw_data(df_max, df_min, shift):
    t_max = df_max["Elapsed Time (s)"].to_numpy(dtype=float)
    t_min = df_min["Elapsed Time (s)"].to_numpy(dtype=float)
    y_max = df_max["Friction Force (N)"].to_numpy(dtype=float)
    y_min = df_min["Friction Force (N)"].to_numpy(dtype=float)

    t_all = np.concatenate([t_max, t_min]) if (len(t_max) + len(t_min)) > 0 else np.array([], dtype=float)
    y_all = np.concatenate([y_max, y_min]) if (len(y_max) + len(y_min)) > 0 else np.array([], dtype=float)

    if len(t_all) == 0:
        return pd.DataFrame({
            "Elapsed Time (s)": [],
            "Absolute Shifted Friction Force (N)": []
        })

    order = np.argsort(t_all)
    t_all = t_all[order]
    y_all = y_all[order]

    y_abs = np.abs(y_all + shift)

    return pd.DataFrame({
        "Elapsed Time (s)": t_all,
        "Absolute Shifted Friction Force (N)": y_abs
    })


def calc_last_fraction_std_from_image5_raw(df_img5_raw, fraction=STD_FRACTION_GLOBAL):
    if df_img5_raw is None or len(df_img5_raw) == 0:
        return np.nan, df_img5_raw.iloc[[]].copy() if df_img5_raw is not None else pd.DataFrame(), 0, 0

    df_sorted = df_img5_raw.sort_values("Elapsed Time (s)").reset_index(drop=True).copy()
    n_total = len(df_sorted)
    n_last = max(1, int(fraction * n_total))

    df_last = df_sorted.iloc[-n_last:].copy()
    std_val = np.std(df_last["Absolute Shifted Friction Force (N)"], ddof=1) if len(df_last) >= 2 else np.nan

    return std_val, df_last, n_total, n_last


def smooth_series_time_based(t, y, smoothing_time=0.1):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(t) < 3 or smoothing_time <= 0:
        return y.copy()

    dt = np.median(np.diff(t))
    if dt <= 0:
        return y.copy()

    fs = 1.0 / dt
    window = int(round(smoothing_time * fs))

    if window < 3:
        return y.copy()

    if window % 2 == 0:
        window += 1

    if window > len(y):
        window = len(y) if len(y) % 2 == 1 else len(y) - 1

    if window < 3:
        return y.copy()

    kernel = np.ones(window) / window
    y_pad = np.pad(y, (window // 2, window // 2), mode="edge")
    return np.convolve(y_pad, kernel, mode="valid")


def build_image12_data(df_max, df_min, shift, smoothing_time):
    t_max = df_max["Elapsed Time (s)"].to_numpy(dtype=float)
    t_min = df_min["Elapsed Time (s)"].to_numpy(dtype=float)
    y_max = df_max["Friction Force (N)"].to_numpy(dtype=float)
    y_min = df_min["Friction Force (N)"].to_numpy(dtype=float)

    t_all = np.concatenate([t_max, t_min])
    y_all = np.concatenate([y_max, y_min])

    order = np.argsort(t_all)
    t_all = t_all[order]
    y_all = y_all[order]

    y_abs = np.abs(y_all + shift)

    if len(t_all) > 6:
        t_all = t_all[:-6]
        y_abs = y_abs[:-6]

    y_smooth = smooth_series_time_based(t_all, y_abs, smoothing_time=smoothing_time)

    df_img12 = pd.DataFrame({
        "Time (s)": t_all,
        "Smoothed Absolute Shifted Friction Force (N)": y_smooth
    })

    if len(df_img12) > 0:
        df_img12["Time (s)"] = df_img12["Time (s)"] - df_img12["Time (s)"].iloc[0]

    return df_img12


def build_image12_combined_figure(df_img11_raw, df_img12, xmax=None, ymax=None):
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(
        df_img11_raw["Elapsed Time (s)"],
        df_img11_raw["Absolute Shifted Friction Force (N)"],
        "o",
        markersize=MARKER_SIZE,
        label="Raw Data"
    )

    ax.plot(
        df_img12["Time (s)"],
        df_img12["Smoothed Absolute Shifted Friction Force (N)"],
        "-",
        linewidth=2,
        label="Smoothed Data"
    )

    raw_xmax = df_img11_raw["Elapsed Time (s)"].max() if len(df_img11_raw) > 0 else 1.0
    smooth_xmax = df_img12["Time (s)"].max() if len(df_img12) > 0 else 1.0
    actual_xmax = max(raw_xmax, smooth_xmax)

    raw_ymax = df_img11_raw["Absolute Shifted Friction Force (N)"].max() if len(df_img11_raw) > 0 else 1.0
    smooth_ymax = df_img12["Smoothed Absolute Shifted Friction Force (N)"].max() if len(df_img12) > 0 else 1.0
    actual_ymax = max(raw_ymax, smooth_ymax)

    if xmax is None or xmax <= 0:
        xmax = actual_xmax
    if ymax is None or ymax <= 0:
        ymax = actual_ymax * 1.05

    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Friction Force (N)")
    ax.set_title("Friction Force Raw Data and Smoothed Data")
    ax.tick_params(direction="in")
    ax.grid(True)
    ax.legend()

    return fig


def redraw_image12_plot():
    with image12_plot_output:
        clear_output(wait=False)

        if image11_raw_global is None or image12_data_global is None:
            return

        fig = build_image12_combined_figure(
            image11_raw_global,
            image12_data_global,
            xmax=image12_xmax_global,
            ymax=image12_ymax_global
        )
        display(fig)
        plt.close(fig)


def build_image13_data(df_img12, normal_load):
    df_img13 = df_img12.copy()
    df_img13["COF"] = df_img13["Smoothed Absolute Shifted Friction Force (N)"] / normal_load
    return df_img13[["Time (s)", "COF"]]


def calculate_average_cof_in_range(df_img13, start_time, end_time):
    if df_img13 is None or len(df_img13) == 0:
        raise ValueError("COF data is empty.")

    if end_time <= start_time:
        raise ValueError("End Time must be greater than Start Time.")

    df_sel = df_img13[
        (df_img13["Time (s)"] >= start_time) &
        (df_img13["Time (s)"] <= end_time)
    ].copy()

    if len(df_sel) == 0:
        raise ValueError("No COF data points found in the selected range.")

    avg_val = df_sel["COF"].mean()
    std_val = df_sel["COF"].std(ddof=1)
    if pd.isna(std_val):
        std_val = 0.0

    return avg_val, std_val, df_sel


def build_image13_figure(df_img13, xmax=None, ymax=None, avg_start=None, avg_end=None, avg_value=None, std_value=None):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df_img13["Time (s)"], df_img13["COF"], linewidth=2, label="COF")

    actual_xmax = df_img13["Time (s)"].max() if len(df_img13) > 0 else 1.0
    actual_ymax = df_img13["COF"].max() if len(df_img13) > 0 else 1.0

    if xmax is None or xmax <= 0:
        xmax = actual_xmax
    if ymax is None or ymax <= 0:
        ymax = actual_ymax * 1.05

    if avg_start is not None and avg_end is not None:
        ax.axvspan(avg_start, avg_end, color="gray", alpha=0.25, label="Average Range")
        ax.axvline(avg_start, linestyle="--", linewidth=1.5, color="black")
        ax.axvline(avg_end, linestyle="--", linewidth=1.5, color="black")

    title_text = "COF Smoothed Data"
    if avg_value is not None and std_value is not None:
        title_text += f" | Average COF (Selected Range) = {avg_value:.6f} ± {std_value:.6f}"

    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("COF")
    ax.set_title(title_text)
    ax.tick_params(direction="in")
    ax.grid(False)

    if avg_start is not None and avg_end is not None:
        ax.legend()

    return fig


def redraw_image13_plot():
    with image13_plot_output:
        clear_output(wait=False)

        if image13_data_global is None:
            return

        fig = build_image13_figure(
            image13_data_global,
            xmax=image13_xmax_global,
            ymax=image13_ymax_global,
            avg_start=image13_avg_start_global,
            avg_end=image13_avg_end_global,
            avg_value=image13_avg_value_global,
            std_value=image13_std_value_global
        )
        display(fig)
        plt.close(fig)


# =========================
# DOWNLOAD HELPERS
# =========================
def dataframe_to_csv_bytes(df):
    return df.to_csv(index=False).encode("utf-8")


def figure_to_png_bytes(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=600, bbox_inches="tight")
    buf.seek(0)
    return buf.read()


def make_download_button(filename, data_bytes, label):
    return widgets.FileDownload(
        data=data_bytes,
        filename=filename,
        description=label,
        button_style="warning"
    )


# =========================
# RESET HELPERS
# =========================
def clear_runtime_globals():
    global df_max_sel_global, df_min_sel_global, mirror_shift_global
    global image12_data_global, image13_data_global
    global image12_xmax_global, image12_ymax_global
    global image13_xmax_global, image13_ymax_global
    global image13_avg_start_global, image13_avg_end_global, image13_avg_value_global, image13_std_value_global
    global image11_raw_global

    df_max_sel_global = None
    df_min_sel_global = None
    mirror_shift_global = 0.0
    image12_data_global = None
    image13_data_global = None

    image12_xmax_global = None
    image12_ymax_global = None
    image13_xmax_global = None
    image13_ymax_global = None

    image13_avg_start_global = None
    image13_avg_end_global = None
    image13_avg_value_global = None
    image13_std_value_global = None

    image11_raw_global = None


def clear_output_widget(widget_obj):
    widget_obj.clear_output(wait=False)


def clear_everything():
    clear_runtime_globals()

    normal_load_box.value = ""
    image12_xmax_box.value = ""
    image12_ymax_box.value = ""
    image13_xmax_box.value = ""
    image13_ymax_box.value = ""
    image13_avg_start_box.value = ""
    image13_avg_end_box.value = ""
    image13_avg_start_box.placeholder = "0"
    image13_avg_end_box.placeholder = ""
    image13_avg_result_box.value = ""
    image13_std_result_box.value = ""

    plt.close("all")

    clear_output_widget(image12_plot_output)
    clear_output_widget(image12_save_output)
    clear_output_widget(image13_plot_output)
    clear_output_widget(image13_save_output)

    image12_section.layout.display = "none"
    image13_section.layout.display = "none"


def reset_progress():
    progress_bar.value = 0
    progress_bar.bar_style = ""
    progress_bar.description = "Frequency scanning:"
    progress_label.value = "0%"

    freq_progress_bar.value = 0
    freq_progress_bar.bar_style = ""
    freq_progress_bar.description = "Frequency tuning:"
    freq_progress_label.value = "0%"


# =========================
# PREPARE FILE
# =========================
def prepare_uploaded_file(file_name, file_bytes):
    global df_trim, begin_info_global, end_info_global
    global sampling_frequency_global, extrema_range_percent_global
    global loaded_filename_global

    df, raw_text = load_sensor_file_from_bytes(file_bytes)
    sampling_frequency_hz = find_sampling_frequency_in_text(raw_text)

    df_trim_local, begin_info, end_info = trim_by_group_rule(df, sampling_frequency_hz)
    df_trim_local["Elapsed Time (s)"] = df_trim_local["Elapsed Time (s)"] - df_trim_local["Elapsed Time (s)"].iloc[0]

    df_trim = df_trim_local
    begin_info_global = begin_info
    end_info_global = end_info
    sampling_frequency_global = sampling_frequency_hz
    extrema_range_percent_global = get_auto_threshold_from_sampling_frequency(sampling_frequency_hz)
    loaded_filename_global = Path(file_name).stem


# =========================
# IMAGE 0
# =========================
def build_image0_data(df, extrema_range_percent=None, progress_bar=None, progress_label=None):
    percentages = np.arange(5, 101, 5, dtype=float)
    rows = []

    if progress_bar is not None:
        progress_bar.description = "Frequency scanning:"
        progress_bar.value = 0
        progress_bar.bar_style = "info"
    if progress_label is not None:
        progress_label.value = "0%"

    for i, pct in enumerate(percentages):
        frac = pct / 100.0

        try:
            freq_detected, _ = detect_frequency_and_phase(df, data_selection_fraction=frac)
            freq_detected = round(freq_detected, 6)

            phase_detected = detect_phase_from_transitions(df, freq_detected, n_cycles=3)

            df_max_det, df_min_det = select_friction_near_ref_extrema(
                df, freq_detected, phase_detected, extrema_range_percent=extrema_range_percent
            )

            mirror_shift_det = compute_mirror_shift(df_max_det, df_min_det)
            df_img5_raw = build_absolute_shifted_raw_data(df_max_det, df_min_det, mirror_shift_det)
            image5_std, _, n_total, n_last = calc_last_fraction_std_from_image5_raw(
                df_img5_raw, fraction=STD_FRACTION_GLOBAL
            )

            rows.append({
                "Percentage Data Selection (%)": pct,
                "Detected Frequency (Hz)": freq_detected,
                "Detected Phase (rad)": phase_detected,
                "Image 5 std": image5_std,
                "Selected points total": n_total,
                f"Selected points last {int(STD_FRACTION_GLOBAL * 100)}% window": n_last
            })
        except Exception:
            rows.append({
                "Percentage Data Selection (%)": pct,
                "Detected Frequency (Hz)": np.nan,
                "Detected Phase (rad)": np.nan,
                "Image 5 std": np.nan,
                "Selected points total": np.nan,
                f"Selected points last {int(STD_FRACTION_GLOBAL * 100)}% window": np.nan
            })

        pct_done = int(round((i + 1) * 100 / len(percentages)))
        if progress_bar is not None:
            progress_bar.value = pct_done
        if progress_label is not None:
            progress_label.value = f"{pct_done}%"

    if progress_bar is not None:
        progress_bar.value = 100
        progress_bar.bar_style = "success"
    if progress_label is not None:
        progress_label.value = "done"

    df_img0 = pd.DataFrame(rows)

    valid = np.isfinite(df_img0["Image 5 std"].to_numpy(dtype=float))
    if np.any(valid):
        best_idx = np.nanargmin(df_img0["Image 5 std"].to_numpy(dtype=float))
        best_pct = float(df_img0.loc[best_idx, "Percentage Data Selection (%)"])
        best_std = float(df_img0.loc[best_idx, "Image 5 std"])
    else:
        best_pct = None
        best_std = None

    return df_img0, best_pct, best_std


def run_image0_for_current_file():
    global image0_data_global, image0_best_percentage_global, image0_best_std_global
    global data_selection_percent_global

    if df_trim is None:
        return

    image0_data_global, image0_best_percentage_global, image0_best_std_global = build_image0_data(
        df_trim,
        extrema_range_percent=extrema_range_percent_global,
        progress_bar=progress_bar,
        progress_label=progress_label
    )

    if image0_best_percentage_global is not None:
        data_selection_percent_global = float(image0_best_percentage_global)
    else:
        data_selection_percent_global = DATA_SELECTION_PERCENT_DEFAULT


# =========================
# FULL ANALYSIS
# =========================
def run_full_analysis_for_current_selection():
    global df_max_sel_global, df_min_sel_global, mirror_shift_global
    global image12_data_global, image13_data_global
    global image13_xmax_global, image13_ymax_global
    global image11_raw_global

    if df_trim is None:
        return

    freq_progress_bar.description = "Frequency tuning:"
    freq_progress_bar.value = 0
    freq_progress_bar.bar_style = "info"
    freq_progress_label.value = "0%"

    image13_data_global = None
    image13_xmax_global = None
    image13_ymax_global = None

    data_selection_fraction = data_selection_percent_global / 100.0

    try:
        freq_detected, _ = detect_frequency_and_phase(df_trim, data_selection_fraction=data_selection_fraction)
        freq_detected = round(freq_detected, 6)

        phase_detected = detect_phase_from_transitions(df_trim, freq_detected, n_cycles=3)

        df_max_det, df_min_det = select_friction_near_ref_extrema(
            df_trim, freq_detected, phase_detected, extrema_range_percent=extrema_range_percent_global
        )

        mirror_shift_det = compute_mirror_shift(df_max_det, df_min_det)
        df_img5_raw_det = build_absolute_shifted_raw_data(df_max_det, df_min_det, mirror_shift_det)
        _image5_std_det, _, _, _ = calc_last_fraction_std_from_image5_raw(
            df_img5_raw_det, fraction=STD_FRACTION_GLOBAL
        )

        scan_low = freq_detected - FREQ_FINE_SCAN_HALF_WIDTH_GLOBAL
        scan_high = freq_detected + FREQ_FINE_SCAN_HALF_WIDTH_GLOBAL
        scan_freqs = np.linspace(scan_low, scan_high, 101)

        scan_stds = []
        scan_phases = []

        for i, f in enumerate(scan_freqs):
            ph = detect_phase_from_transitions(df_trim, f, n_cycles=3)
            df_max_f, df_min_f = select_friction_near_ref_extrema(
                df_trim, f, ph, extrema_range_percent=extrema_range_percent_global
            )
            shift_f = compute_mirror_shift(df_max_f, df_min_f)
            df_img5_f = build_absolute_shifted_raw_data(df_max_f, df_min_f, shift_f)
            std_f, _, _, _ = calc_last_fraction_std_from_image5_raw(df_img5_f, fraction=STD_FRACTION_GLOBAL)

            scan_phases.append(ph)
            scan_stds.append(std_f)

            pct = int(round((i + 1) * 100 / len(scan_freqs)))
            freq_progress_bar.value = pct
            freq_progress_label.value = f"{pct}%"

        scan_stds = np.array(scan_stds, dtype=float)
        scan_phases = np.array(scan_phases, dtype=float)

        valid = np.isfinite(scan_stds)
        if not np.any(valid):
            raise ValueError("No valid Image 5 std values found in frequency scan.")

        valid_indices = np.where(valid)[0]
        best_local_idx = np.argmin(scan_stds[valid])
        best_idx = valid_indices[best_local_idx]

        selected_frequency = scan_freqs[best_idx]
        selected_phase = scan_phases[best_idx]

        df_max_sel, df_min_sel = select_friction_near_ref_extrema(
            df_trim, selected_frequency, selected_phase, extrema_range_percent=extrema_range_percent_global
        )

        mirror_shift = compute_mirror_shift(df_max_sel, df_min_sel)
        df_img11_raw = build_absolute_shifted_raw_data(df_max_sel, df_min_sel, mirror_shift)

        df_max_sel_global = df_max_sel.copy()
        df_min_sel_global = df_min_sel.copy()
        mirror_shift_global = mirror_shift
        image11_raw_global = df_img11_raw.copy()

        image12_data_global = build_image12_data(
            df_max_sel_global,
            df_min_sel_global,
            mirror_shift_global,
            smoothing_time=SMOOTHING_IMAGE12
        )

        freq_progress_bar.value = 100
        freq_progress_bar.bar_style = "success"
        freq_progress_label.value = "done"

    except Exception as e:
        with image12_save_output:
            clear_output(wait=False)
            print("Error:", e)
        freq_progress_bar.bar_style = "danger"
        freq_progress_label.value = "error"
        return

    redraw_image12_plot()
    image12_section.layout.display = ""
    image13_section.layout.display = ""


# =========================
# UI
# =========================
upload_widget = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    button_style="primary"
)

file_name_label = widgets.HTML(
    value="<span style='color:gray;'>No file selected</span>",
    layout=widgets.Layout(width="500px")
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Frequency scanning:",
    bar_style="",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

progress_label = widgets.HTML(
    "0%",
    layout=widgets.Layout(width="50px", margin="0 0 0 10px")
)

freq_progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Frequency tuning:",
    bar_style="",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

freq_progress_label = widgets.HTML(
    "0%",
    layout=widgets.Layout(width="50px", margin="0 0 0 10px")
)

image12_xmax_box = widgets.Text(
    value="",
    description="Max X:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="150px")
)

image12_ymax_box = widgets.Text(
    value="",
    description="Max Y:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="150px")
)

image12_apply_axis_button = widgets.Button(description="Apply Axis", button_style="success")
image12_reset_axis_button = widgets.Button(description="Reset Axis")
image12_save_raw_button = widgets.Button(description="Save Raw Data", button_style="warning")
image12_save_smoothed_button = widgets.Button(description="Save Smoothed Data", button_style="warning")
image12_save_image_button = widgets.Button(description="Save Image")

normal_load_box = widgets.Text(
    value="",
    description="Normal Load (N):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="220px")
)

normal_load_apply_button = widgets.Button(description="Apply", button_style="success")
normal_load_reset_button = widgets.Button(description="Reset")

image13_avg_start_box = widgets.Text(
    value="",
    placeholder="0",
    description="Start (s):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="170px")
)

image13_avg_end_box = widgets.Text(
    value="",
    placeholder="",
    description="End (s):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="170px")
)

image13_avg_apply_button = widgets.Button(description="Apply Avg", button_style="success")
image13_avg_reset_button = widgets.Button(description="Reset Avg")

image13_avg_result_box = widgets.Text(
    value="",
    description="Avg COF:",
    layout=widgets.Layout(width="250px"),
    style={"description_width": "initial"},
    disabled=True
)

image13_std_result_box = widgets.Text(
    value="",
    description="±",
    layout=widgets.Layout(width="200px"),
    style={"description_width": "initial"},
    disabled=True
)

image13_xmax_box = widgets.Text(
    value="",
    description="Max X:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="150px")
)

image13_ymax_box = widgets.Text(
    value="",
    description="Max Y:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="150px")
)

image13_apply_axis_button = widgets.Button(description="Apply Axis", button_style="success")
image13_reset_axis_button = widgets.Button(description="Reset Axis")
image13_save_data_button = widgets.Button(description="Save Data", button_style="warning")
image13_save_image_button = widgets.Button(description="Save Image")

image12_plot_output = widgets.Output()
image12_save_output = widgets.Output()
image13_plot_output = widgets.Output()
image13_save_output = widgets.Output()

image12_controls = widgets.HBox([
    image12_xmax_box, image12_ymax_box,
    image12_apply_axis_button, image12_reset_axis_button,
    image12_save_raw_button, image12_save_smoothed_button, image12_save_image_button
])

image12_section = widgets.VBox([
    image12_plot_output,
    image12_controls,
    image12_save_output
])
image12_section.layout.display = "none"

image13_avg_controls = widgets.HBox([
    image13_avg_start_box,
    image13_avg_end_box,
    image13_avg_apply_button,
    image13_avg_reset_button
])

image13_avg_result_controls = widgets.HBox([
    image13_avg_result_box,
    image13_std_result_box
])

image13_controls = widgets.HBox([
    image13_xmax_box, image13_ymax_box,
    image13_apply_axis_button, image13_reset_axis_button,
    image13_save_data_button, image13_save_image_button
])

image13_section = widgets.VBox([
    widgets.HTML("<hr><b>COF Image controls</b>"),
    widgets.HBox([normal_load_box, normal_load_apply_button, normal_load_reset_button]),
    image13_avg_controls,
    image13_avg_result_controls,
    image13_plot_output,
    image13_controls,
    image13_save_output
])
image13_section.layout.display = "none"


# =========================
# ACTIONS
# =========================
def upload_changed(change):
    global image0_data_global, image0_best_percentage_global, image0_best_std_global
    global df_trim, begin_info_global, end_info_global, sampling_frequency_global, extrema_range_percent_global
    global data_selection_percent_global

    if not upload_widget.value:
        return

    try:
        uploaded_value = upload_widget.value

        if isinstance(uploaded_value, dict):
            first_key = list(uploaded_value.keys())[0]
            file_info = uploaded_value[first_key]
            file_name = file_info["metadata"]["name"] if "metadata" in file_info else first_key
            file_bytes = file_info["content"]
        else:
            file_info = uploaded_value[0]
            file_name = file_info["name"]
            file_bytes = file_info["content"]

        df_trim = None
        begin_info_global = None
        end_info_global = None
        sampling_frequency_global = None
        extrema_range_percent_global = 0.01

        image0_data_global = None
        image0_best_percentage_global = None
        image0_best_std_global = None

        data_selection_percent_global = DATA_SELECTION_PERCENT_DEFAULT

        reset_progress()
        clear_everything()

        file_name_label.value = f"<b>File:</b> {file_name}"

        prepare_uploaded_file(file_name, file_bytes)
        run_image0_for_current_file()
        run_full_analysis_for_current_selection()

    except Exception as e:
        file_name_label.value = f"<span style='color:red;'>Error: {str(e)}</span>"
        progress_bar.bar_style = "danger"
        progress_label.value = "error"


def image12_apply_axis_clicked(_):
    global image12_xmax_global, image12_ymax_global

    try:
        x_text = image12_xmax_box.value.strip()
        y_text = image12_ymax_box.value.strip()

        image12_xmax_global = None if x_text == "" else float(x_text)
        image12_ymax_global = None if y_text == "" else float(y_text)

        if image12_xmax_global is not None and image12_xmax_global <= 0:
            image12_xmax_global = None
        if image12_ymax_global is not None and image12_ymax_global <= 0:
            image12_ymax_global = None
    except Exception:
        with image12_save_output:
            clear_output(wait=False)
            print("Please enter valid numbers for Max X / Max Y.")
        return

    redraw_image12_plot()


def image12_reset_axis_clicked(_):
    global image12_xmax_global, image12_ymax_global
    image12_xmax_global = None
    image12_ymax_global = None
    image12_xmax_box.value = ""
    image12_ymax_box.value = ""
    redraw_image12_plot()


def image12_save_raw_clicked(_):
    clear_output_widget(image12_save_output)

    with image12_save_output:
        if image11_raw_global is None:
            print("Raw data is not available.")
            return

        default_name = f"{loaded_filename_global}_Raw_Data.csv" if loaded_filename_global else "Raw_Data.csv"
        display(make_download_button(default_name, dataframe_to_csv_bytes(image11_raw_global), "Download Raw Data"))


def image12_save_smoothed_clicked(_):
    clear_output_widget(image12_save_output)

    with image12_save_output:
        if image12_data_global is None:
            print("Smoothed data is not available.")
            return

        default_name = f"{loaded_filename_global}_Smoothed_Data.csv" if loaded_filename_global else "Smoothed_Data.csv"
        display(make_download_button(default_name, dataframe_to_csv_bytes(image12_data_global), "Download Smoothed Data"))


def image12_save_image_clicked(_):
    clear_output_widget(image12_save_output)

    with image12_save_output:
        if image11_raw_global is None or image12_data_global is None:
            print("Combined image is not available.")
            return

        fig = build_image12_combined_figure(
            image11_raw_global,
            image12_data_global,
            xmax=image12_xmax_global,
            ymax=image12_ymax_global
        )
        png_bytes = figure_to_png_bytes(fig)
        plt.close(fig)

        default_name = f"{loaded_filename_global}_Raw_and_Smoothed_Data.png" if loaded_filename_global else "Raw_and_Smoothed_Data.png"
        display(make_download_button(default_name, png_bytes, "Download Image"))


def normal_load_apply_clicked(_):
    global image13_data_global, image13_xmax_global, image13_ymax_global
    global image13_avg_start_global, image13_avg_end_global, image13_avg_value_global, image13_std_value_global

    clear_output_widget(image13_save_output)

    if image12_data_global is None:
        return

    try:
        normal_load = float(normal_load_box.value)
    except Exception:
        with image13_save_output:
            print("Please enter a valid Normal Load.")
        return

    if normal_load == 0:
        with image13_save_output:
            print("Normal Load cannot be 0.")
        return

    image13_data_global = build_image13_data(image12_data_global, normal_load)
    image13_xmax_global = None
    image13_ymax_global = None

    image13_avg_start_global = None
    image13_avg_end_global = None
    image13_avg_value_global = None
    image13_std_value_global = None

    image13_xmax_box.value = ""
    image13_ymax_box.value = ""
    image13_avg_start_box.value = ""
    image13_avg_end_box.value = ""
    image13_avg_result_box.value = ""
    image13_std_result_box.value = ""

    image13_avg_start_box.placeholder = "0"
    if image13_data_global is not None and len(image13_data_global) > 0:
        image13_avg_end_box.placeholder = f"{float(image13_data_global['Time (s)'].max()):.3f}"
    else:
        image13_avg_end_box.placeholder = ""

    redraw_image13_plot()


def normal_load_reset_clicked(_):
    global image13_data_global, image13_xmax_global, image13_ymax_global
    global image13_avg_start_global, image13_avg_end_global, image13_avg_value_global, image13_std_value_global

    normal_load_box.value = ""
    image13_data_global = None
    image13_xmax_global = None
    image13_ymax_global = None

    image13_avg_start_global = None
    image13_avg_end_global = None
    image13_avg_value_global = None
    image13_std_value_global = None

    image13_xmax_box.value = ""
    image13_ymax_box.value = ""
    image13_avg_start_box.value = ""
    image13_avg_end_box.value = ""
    image13_avg_start_box.placeholder = "0"
    image13_avg_end_box.placeholder = ""
    image13_avg_result_box.value = ""
    image13_std_result_box.value = ""

    clear_output_widget(image13_plot_output)
    clear_output_widget(image13_save_output)


def image13_avg_apply_clicked(_):
    global image13_avg_start_global, image13_avg_end_global, image13_avg_value_global, image13_std_value_global

    clear_output_widget(image13_save_output)

    if image13_data_global is None:
        with image13_save_output:
            print("Please apply Normal Load first.")
        return

    try:
        start_text = image13_avg_start_box.value.strip()
        end_text = image13_avg_end_box.value.strip()

        if start_text == "":
            start_text = image13_avg_start_box.placeholder.strip()
        if end_text == "":
            end_text = image13_avg_end_box.placeholder.strip()

        start_val = float(start_text)
        end_val = float(end_text)
    except Exception:
        with image13_save_output:
            print("Please enter valid numbers for Start and End.")
        return

    try:
        avg_val, std_val, _ = calculate_average_cof_in_range(image13_data_global, start_val, end_val)

        image13_avg_start_global = start_val
        image13_avg_end_global = end_val
        image13_avg_value_global = avg_val
        image13_std_value_global = std_val

        image13_avg_result_box.value = f"{avg_val:.6f}"
        image13_std_result_box.value = f"{std_val:.6f}"

        redraw_image13_plot()
    except Exception as e:
        with image13_save_output:
            print("Error:", e)


def image13_avg_reset_clicked(_):
    global image13_avg_start_global, image13_avg_end_global, image13_avg_value_global, image13_std_value_global

    image13_avg_start_global = None
    image13_avg_end_global = None
    image13_avg_value_global = None
    image13_std_value_global = None

    image13_avg_start_box.value = ""
    image13_avg_end_box.value = ""
    image13_avg_result_box.value = ""
    image13_std_result_box.value = ""

    clear_output_widget(image13_save_output)
    redraw_image13_plot()


def image13_apply_axis_clicked(_):
    global image13_xmax_global, image13_ymax_global

    try:
        x_text = image13_xmax_box.value.strip()
        y_text = image13_ymax_box.value.strip()

        image13_xmax_global = None if x_text == "" else float(x_text)
        image13_ymax_global = None if y_text == "" else float(y_text)

        if image13_xmax_global is not None and image13_xmax_global <= 0:
            image13_xmax_global = None
        if image13_ymax_global is not None and image13_ymax_global <= 0:
            image13_ymax_global = None
    except Exception:
        with image13_save_output:
            clear_output(wait=False)
            print("Please enter valid numbers for Max X / Max Y.")
        return

    redraw_image13_plot()


def image13_reset_axis_clicked(_):
    global image13_xmax_global, image13_ymax_global
    image13_xmax_global = None
    image13_ymax_global = None
    image13_xmax_box.value = ""
    image13_ymax_box.value = ""
    redraw_image13_plot()


def image13_save_data_clicked(_):
    clear_output_widget(image13_save_output)

    with image13_save_output:
        if image13_data_global is None:
            print("Please apply Normal Load first.")
            return

        default_name = f"{loaded_filename_global}_Image13_COF_Data.csv" if loaded_filename_global else "Image13_COF_Data.csv"
        display(make_download_button(default_name, dataframe_to_csv_bytes(image13_data_global), "Download COF Data"))


def image13_save_image_clicked(_):
    clear_output_widget(image13_save_output)

    with image13_save_output:
        if image13_data_global is None:
            print("Please apply Normal Load first.")
            return

        fig = build_image13_figure(
            image13_data_global,
            xmax=image13_xmax_global,
            ymax=image13_ymax_global,
            avg_start=image13_avg_start_global,
            avg_end=image13_avg_end_global,
            avg_value=image13_avg_value_global,
            std_value=image13_std_value_global
        )
        png_bytes = figure_to_png_bytes(fig)
        plt.close(fig)

        default_name = f"{loaded_filename_global}_Image13_COF_Plot.png" if loaded_filename_global else "Image13_COF_Plot.png"
        display(make_download_button(default_name, png_bytes, "Download COF Image"))


# =========================
# CONNECT
# =========================
upload_widget.observe(upload_changed, names="value")

image12_apply_axis_button.on_click(image12_apply_axis_clicked)
image12_reset_axis_button.on_click(image12_reset_axis_clicked)
image12_save_raw_button.on_click(image12_save_raw_clicked)
image12_save_smoothed_button.on_click(image12_save_smoothed_clicked)
image12_save_image_button.on_click(image12_save_image_clicked)

normal_load_apply_button.on_click(normal_load_apply_clicked)
normal_load_reset_button.on_click(normal_load_reset_clicked)

image13_avg_apply_button.on_click(image13_avg_apply_clicked)
image13_avg_reset_button.on_click(image13_avg_reset_clicked)

image13_apply_axis_button.on_click(image13_apply_axis_clicked)
image13_reset_axis_button.on_click(image13_reset_axis_clicked)
image13_save_data_button.on_click(image13_save_data_clicked)
image13_save_image_button.on_click(image13_save_image_clicked)


# =========================
# LAYOUT
# =========================
display(widgets.VBox([
    widgets.HBox([upload_widget, file_name_label]),
    widgets.HBox([progress_bar, progress_label], layout=widgets.Layout(align_items="center")),
    widgets.HBox([freq_progress_bar, freq_progress_label], layout=widgets.Layout(align_items="center")),
    image12_section,
    image13_section
]))